In [1]:
import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Any, Optional

import pandas as pd
import requests
from openai import OpenAI

In [2]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

BASE_DIRECTORY = Path.cwd()

COMPANIES_CSV = BASE_DIRECTORY / "sp500.csv"
QUESTIONS_DIRECTORY = BASE_DIRECTORY / "questions"
ANSWERS_DIRECTORY = BASE_DIRECTORY / "answers"
SNAPSHOTS_DIRECTORY = BASE_DIRECTORY / "finnhub_snapshots"

# ============================================================
# LLM CONFIGURATION
# ============================================================

LLM_BASE_URL = "http://api.llm.apps.os.dcs.gla.ac.uk/v1"

# Use this instead when outside the University of Glasgow network:
# LLM_BASE_URL = "http://api.terrier.org/v1"

LLM_MODEL = "gpt-oss-120b"

# ============================================================
# FINNHUB CONFIGURATION
# ============================================================

FINNHUB_BASE_URL = "https://finnhub.io/api/v1"

# Number of recent calendar days for company news.
NEWS_LOOKBACK_DAYS = 7


# Sleeping to stay below 60 calls per minute limit.
FINNHUB_REQUEST_DELAY = 1.1

MAX_FINNHUB_RETRIES = 5
MAX_LLM_RETRIES = 3

# Set to None to process every company.
COMPANY_LIMIT = None

# Set True to overwrite an answer files that already exist.
OVERWRITE_EXISTING_ANSWERS = False

In [3]:
def require_environment_variable(variable_name: str) -> str:
    value = os.environ.get(variable_name)

    if not value:
        raise EnvironmentError(
            f"The environment variable {variable_name!r} is not set.\n"
            f"Set it before running the notebook."
        )

    return value


FINNHUB_API_KEY = "d95tgvpr01qj66kpqv40d95tgvpr01qj66kpqv4g"
IDA_LLM_API_KEY = os.environ['IDA_LLM_API_KEY']
llm_client = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=IDA_LLM_API_KEY
)

print("API configuration loaded successfully.")

API configuration loaded successfully.


In [4]:
def safe_filename(value: str) -> str:
    #Convert a company name into a filesystem-safe filename while preserving spaces and other readable characters where possible.
    value = str(value).strip()

    # Remove characters that are invalid in Windows/Linux filenames.
    value = re.sub(r'[<>:"/\\|?*]', "_", value)

    # Remove repeated whitespace.
    value = re.sub(r"\s+", " ", value)

    # Avoid trailing periods and spaces.
    return value.rstrip(". ")


def normalise_company_name(value: str) -> str:
    
    #Normalised representation used when matching question filenames.
    value = str(value).lower().strip()
    value = value.replace("&", "and")
    value = re.sub(r"[^a-z0-9]+", "", value)
    return value


def serialise_json(data: Any) -> str:
    return json.dumps(
        data,
        indent=2,
        ensure_ascii=False,
        default=str
    )


def utc_timestamp_to_text(timestamp: Any) -> Optional[str]:

    #Convert a Unix timestamp to a readable UTC timestamp.
    try:
        timestamp = int(timestamp)

        if timestamp <= 0:
            return None

        return datetime.fromtimestamp(
            timestamp,
            tz=timezone.utc
        ).isoformat()
    except (TypeError, ValueError, OSError):
        return None

In [5]:
def build_question_file_index(
    questions_directory: Path
) -> dict[str, Path]:
   
    #Create an index of normalised question filenames.
   
    if not questions_directory.exists():
        raise FileNotFoundError(
            f"Questions directory does not exist: {questions_directory}"
        )

    question_files = list(questions_directory.glob("*.txt"))

    if not question_files:
        raise FileNotFoundError(
            f"No .txt question files were found in {questions_directory}"
        )

    index = {}

    for path in question_files:
        normalised_stem = normalise_company_name(path.stem)
        index[normalised_stem] = path

    return index


def find_question_file(
    company_name: str,
    ticker: str,
    question_file_index: dict[str, Path]
) -> Optional[Path]:
   
    #Find a question file using the company name or ticker.
    possible_names = [
        company_name,
        safe_filename(company_name),
        ticker,
        f"{company_name} {ticker}",
        f"{ticker} {company_name}"
    ]

    for possible_name in possible_names:
        key = normalise_company_name(possible_name)

        if key in question_file_index:
            return question_file_index[key]

    # Partial matching fallback.
    company_key = normalise_company_name(company_name)

    partial_matches = [
        path
        for key, path in question_file_index.items()
        if company_key in key or key in company_key
    ]

    if len(partial_matches) == 1:
        return partial_matches[0]

    return None


def load_questions(question_file: Path) -> list[str]:
    
    #Load questions from a text file.
    text = question_file.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()

    if not text:
        raise ValueError(f"Question file is empty: {question_file}")

    # Match numbered questions, including multiline questions.
    numbered_pattern = re.compile(
        r"(?:^|\n)\s*(?:question\s*)?(\d{1,3})\s*[\.\):\-]\s*"
        r"(.*?)(?=(?:\n\s*(?:question\s*)?\d{1,3}\s*[\.\):\-]\s*)|\Z)",
        flags=re.IGNORECASE | re.DOTALL
    )

    matches = numbered_pattern.findall(text)

    if matches:
        questions = [
            re.sub(r"\s+", " ", question_text).strip()
            for _, question_text in matches
            if question_text.strip()
        ]
    else:
        questions = [
            re.sub(r"\s+", " ", line).strip()
            for line in text.splitlines()
            if line.strip()
        ]

    if not questions:
        raise ValueError(
            f"No questions could be extracted from {question_file}"
        )

    return questions

In [6]:
finnhub_session = requests.Session()


def finnhub_get(
    endpoint: str,
    params: Optional[dict[str, Any]] = None
) -> Any:
    
    #Send a rate-limited Finnhub request with retry handling.
    params = dict(params or {})
    params["token"] = FINNHUB_API_KEY

    url = f"{FINNHUB_BASE_URL}/{endpoint.lstrip('/')}"

    for attempt in range(1, MAX_FINNHUB_RETRIES + 1):
        try:
            response = finnhub_session.get(
                url,
                params=params,
                timeout=45
            )

            if response.status_code == 429:
                wait_time = min(60, 10 * attempt)

                print(
                    f"Finnhub rate limit reached. "
                    f"Retrying after {wait_time} seconds..."
                )

                time.sleep(wait_time)
                continue

            response.raise_for_status()

            data = response.json()

            # Throttle successful requests.
            time.sleep(FINNHUB_REQUEST_DELAY)

            return data

        except requests.RequestException as error:
            if attempt == MAX_FINNHUB_RETRIES:
                raise RuntimeError(
                    f"Finnhub request failed for endpoint {endpoint}: {error}"
                ) from error

            wait_time = min(
                60,
                (2 ** attempt) + random.uniform(0, 1)
            )

            print(
                f"Finnhub request error for {endpoint}. "
                f"Retry {attempt}/{MAX_FINNHUB_RETRIES} "
                f"after {wait_time:.1f} seconds."
            )

            time.sleep(wait_time)

        except ValueError as error:
            raise RuntimeError(
                f"Finnhub returned invalid JSON for endpoint {endpoint}"
            ) from error

    raise RuntimeError(f"Finnhub request failed: {endpoint}")

In [7]:
IMPORTANT_METRICS = [
    "10DayAverageTradingVolume",
    "13WeekPriceReturnDaily",
    "26WeekPriceReturnDaily",
    "3MonthAverageTradingVolume",
    "52WeekHigh",
    "52WeekHighDate",
    "52WeekLow",
    "52WeekLowDate",
    "52WeekPriceReturnDaily",
    "5DayPriceReturnDaily",
    "assetTurnoverAnnual",
    "beta",
    "bookValuePerShareAnnual",
    "cashFlowPerShareAnnual",
    "currentRatioAnnual",
    "dividendGrowthRate5Y",
    "dividendPerShareAnnual",
    "dividendYieldIndicatedAnnual",
    "ebitdPerShareAnnual",
    "epsAnnual",
    "epsGrowth3Y",
    "epsGrowth5Y",
    "epsGrowthQuarterlyYoy",
    "grossMarginAnnual",
    "inventoryTurnoverAnnual",
    "longTermDebtEquityAnnual",
    "marketCapitalization",
    "netDebtAnnual",
    "netProfitMarginAnnual",
    "operatingMarginAnnual",
    "payoutRatioAnnual",
    "pbAnnual",
    "pcfShareAnnual",
    "peAnnual",
    "peBasicExclExtraTTM",
    "peExclExtraAnnual",
    "priceRelativeToS&P50013Week",
    "priceRelativeToS&P50026Week",
    "priceRelativeToS&P5004Week",
    "priceRelativeToS&P50052Week",
    "psAnnual",
    "quickRatioAnnual",
    "receivablesTurnoverAnnual",
    "revenueGrowth3Y",
    "revenueGrowth5Y",
    "revenueGrowthQuarterlyYoy",
    "revenuePerShareAnnual",
    "roaRfy",
    "roaTTM",
    "roeRfy",
    "roeTTM",
    "roiAnnual",
    "salesPerShareAnnual",
    "totalDebtEquityAnnual"
]


def filter_basic_financials(data: Any) -> dict[str, Any]:
    if not isinstance(data, dict):
        return {}

    metrics = data.get("metric", {})

    selected_metrics = {
        metric_name: metrics.get(metric_name)
        for metric_name in IMPORTANT_METRICS
        if metric_name in metrics
    }

    # Keep recent time-series entries if Finnhub provides them.
    filtered_series = {}

    series = data.get("series", {})

    if isinstance(series, dict):
        for period_name, period_values in series.items():
            if not isinstance(period_values, dict):
                continue

            filtered_series[period_name] = {}

            for metric_name, values in period_values.items():
                if isinstance(values, list):
                    filtered_series[period_name][metric_name] = values[-8:]
                else:
                    filtered_series[period_name][metric_name] = values

    return {
        "symbol": data.get("symbol"),
        "metric_type": data.get("metricType"),
        "metrics": selected_metrics,
        "recent_series": filtered_series
    }


def clean_news(news_items: Any, maximum_items: int = 20) -> list[dict]:
    if not isinstance(news_items, list):
        return []

    cleaned = []

    for item in news_items[:maximum_items]:
        if not isinstance(item, dict):
            continue

        cleaned.append({
            "datetime_utc": utc_timestamp_to_text(item.get("datetime")),
            "headline": item.get("headline"),
            "summary": item.get("summary"),
            "source": item.get("source"),
            "category": item.get("category"),
            "related": item.get("related"),
            "url": item.get("url")
        })

    return cleaned


def clean_recommendations(data: Any) -> list[dict]:
    if not isinstance(data, list):
        return []

    return [
        {
            "period": row.get("period"),
            "strong_buy": row.get("strongBuy"),
            "buy": row.get("buy"),
            "hold": row.get("hold"),
            "sell": row.get("sell"),
            "strong_sell": row.get("strongSell")
        }
        for row in data[:12]
        if isinstance(row, dict)
    ]


def clean_earnings(data: Any) -> list[dict]:
    if not isinstance(data, list):
        return []

    return [
        {
            "period": row.get("period"),
            "quarter": row.get("quarter"),
            "year": row.get("year"),
            "actual_eps": row.get("actual"),
            "estimated_eps": row.get("estimate"),
            "surprise": row.get("surprise"),
            "surprise_percent": row.get("surprisePercent"),
            "symbol": row.get("symbol")
        }
        for row in data[:8]
        if isinstance(row, dict)
    ]

In [8]:
def fetch_finnhub_snapshot(
    company_name: str,
    ticker: str,
    category: str,
    snapshot_date: str
) -> dict[str, Any]:
    
    #Fetch a daily information snapshot for one company.
    today = datetime.now().date()
    news_start = today - timedelta(days=NEWS_LOOKBACK_DAYS)

    snapshot = {
        "snapshot_date": snapshot_date,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "company_name": company_name,
        "ticker": ticker,
        "category": category,
        "data": {},
        "endpoint_errors": {}
    }

    endpoint_requests = {
        "quote": (
            "quote",
            {"symbol": ticker}
        ),
        "company_profile": (
            "stock/profile2",
            {"symbol": ticker}
        ),
        "basic_financials": (
            "stock/metric",
            {
                "symbol": ticker,
                "metric": "all"
            }
        ),
        "company_news": (
            "company-news",
            {
                "symbol": ticker,
                "from": news_start.isoformat(),
                "to": today.isoformat()
            }
        ),
        "recommendation_trends": (
            "stock/recommendation",
            {"symbol": ticker}
        ),
        "earnings_surprises": (
            "stock/earnings",
            {
                "symbol": ticker,
                "limit": 8
            }
        )
    }

    for data_name, (endpoint, params) in endpoint_requests.items():
        try:
            print(f"    Fetching {data_name}...")

            result = finnhub_get(
                endpoint=endpoint,
                params=params
            )

            if data_name == "basic_financials":
                result = filter_basic_financials(result)
            elif data_name == "company_news":
                result = clean_news(result)
            elif data_name == "recommendation_trends":
                result = clean_recommendations(result)
            elif data_name == "earnings_surprises":
                result = clean_earnings(result)

            snapshot["data"][data_name] = result

        except Exception as error:
            print(f"    Warning: {data_name} failed: {error}")

            snapshot["endpoint_errors"][data_name] = str(error)
            snapshot["data"][data_name] = None

    quote = snapshot["data"].get("quote")

    if isinstance(quote, dict):
        quote["timestamp_utc"] = utc_timestamp_to_text(quote.get("t"))

        # Add readable field names because Finnhub quote keys are abbreviated.
        snapshot["data"]["quote_interpreted"] = {
            "current_price": quote.get("c"),
            "daily_change": quote.get("d"),
            "daily_percent_change": quote.get("dp"),
            "daily_high": quote.get("h"),
            "daily_low": quote.get("l"),
            "daily_open": quote.get("o"),
            "previous_close": quote.get("pc"),
            "market_timestamp_utc": quote.get("timestamp_utc")
        }

    return snapshot

In [9]:
def snapshot_file_path(
    snapshot_date: str,
    company_name: str,
    ticker: str
) -> Path:
    directory = SNAPSHOTS_DIRECTORY / snapshot_date

    filename = (
        f"{safe_filename(company_name)}_"
        f"{safe_filename(ticker)}_snapshot.json"
    )

    return directory / filename


def save_snapshot(snapshot: dict[str, Any]) -> Path:
    path = snapshot_file_path(
        snapshot_date=snapshot["snapshot_date"],
        company_name=snapshot["company_name"],
        ticker=snapshot["ticker"]
    )

    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(
        serialise_json(snapshot),
        encoding="utf-8"
    )

    return path


def find_previous_snapshot(
    company_name: str,
    ticker: str,
    current_date: str
) -> Optional[dict[str, Any]]:
   
    #Find the most recent stored snapshot before current_date.
   
    if not SNAPSHOTS_DIRECTORY.exists():
        return None

    current_date_object = datetime.strptime(
        current_date,
        "%Y-%m-%d"
    ).date()

    candidate_files = []

    for date_directory in SNAPSHOTS_DIRECTORY.iterdir():
        if not date_directory.is_dir():
            continue

        try:
            directory_date = datetime.strptime(
                date_directory.name,
                "%Y-%m-%d"
            ).date()
        except ValueError:
            continue

        if directory_date >= current_date_object:
            continue

        candidate_path = snapshot_file_path(
            snapshot_date=date_directory.name,
            company_name=company_name,
            ticker=ticker
        )

        if candidate_path.exists():
            candidate_files.append(
                (directory_date, candidate_path)
            )

    if not candidate_files:
        return None

    _, newest_file = max(
        candidate_files,
        key=lambda item: item[0]
    )

    try:
        return json.loads(
            newest_file.read_text(encoding="utf-8")
        )
    except (json.JSONDecodeError, OSError):
        return None

In [10]:
def percentage_change(
    current_value: Any,
    previous_value: Any
) -> Optional[float]:
    try:
        current_number = float(current_value)
        previous_number = float(previous_value)

        if previous_number == 0:
            return None

        return (
            (current_number - previous_number)
            / abs(previous_number)
        ) * 100

    except (TypeError, ValueError):
        return None


def compare_snapshots(
    current_snapshot: dict[str, Any],
    previous_snapshot: Optional[dict[str, Any]]
) -> dict[str, Any]:
    
    #Produce an explicit daily comparison for the LLM.
    
    if previous_snapshot is None:
        return {
            "comparison_available": False,
            "message": (
                "No earlier local snapshot was found. This is likely the "
                "first run for this company. Use the current quote's daily "
                "change and daily percentage change fields."
            )
        }

    current_data = current_snapshot.get("data", {})
    previous_data = previous_snapshot.get("data", {})

    current_quote = current_data.get("quote_interpreted") or {}
    previous_quote = previous_data.get("quote_interpreted") or {}

    current_metrics = (
        (current_data.get("basic_financials") or {}).get("metrics", {})
    )

    previous_metrics = (
        (previous_data.get("basic_financials") or {}).get("metrics", {})
    )

    metric_changes = {}

    all_metric_names = sorted(
        set(current_metrics) | set(previous_metrics)
    )

    for metric_name in all_metric_names:
        current_value = current_metrics.get(metric_name)
        previous_value = previous_metrics.get(metric_name)

        if current_value != previous_value:
            metric_changes[metric_name] = {
                "previous": previous_value,
                "current": current_value,
                "percentage_change": percentage_change(
                    current_value,
                    previous_value
                )
            }

    current_news = current_data.get("company_news") or []
    previous_news = previous_data.get("company_news") or []

    previous_news_keys = {
        (
            item.get("headline"),
            item.get("datetime_utc")
        )
        for item in previous_news
        if isinstance(item, dict)
    }

    new_news = [
        item
        for item in current_news
        if isinstance(item, dict)
        and (
            item.get("headline"),
            item.get("datetime_utc")
        ) not in previous_news_keys
    ]

    return {
        "comparison_available": True,
        "previous_snapshot_date": previous_snapshot.get("snapshot_date"),
        "current_snapshot_date": current_snapshot.get("snapshot_date"),
        "price_comparison": {
            "previous_snapshot_current_price":
                previous_quote.get("current_price"),
            "current_snapshot_current_price":
                current_quote.get("current_price"),
            "change_between_snapshots": (
                float(current_quote["current_price"])
                - float(previous_quote["current_price"])
                if current_quote.get("current_price") is not None
                and previous_quote.get("current_price") is not None
                else None
            ),
            "percentage_change_between_snapshots": percentage_change(
                current_quote.get("current_price"),
                previous_quote.get("current_price")
            ),
            "current_market_daily_change":
                current_quote.get("daily_change"),
            "current_market_daily_percent_change":
                current_quote.get("daily_percent_change")
        },
        "changed_financial_metrics": metric_changes,
        "new_company_news_since_previous_snapshot": new_news
    }

In [11]:
def format_questions_for_prompt(
    questions: list[str]
) -> str:
    return "\n\n".join(
        f"{index}. {question}"
        for index, question in enumerate(questions, start=1)
    )


def build_company_prompt(
    company_name: str,
    ticker: str,
    category: str,
    questions: list[str],
    current_snapshot: dict[str, Any],
    daily_comparison: dict[str, Any]
) -> str:
    questions_text = format_questions_for_prompt(questions)

    return f"""
You are a financial research assistant participating in a longitudinal experiment that studies whether daily changes in company information may help predict future stock performance.

Your task is to answer every supplied question for the specified company using the available Finnhub data, recent Finnhub news, historical financial metrics, the question’s factual context, and your established general knowledge of the company.

COMPANY INFORMATION

Company name: {company_name}
Ticker: {ticker}
Category: {category}
Analysis date: {current_snapshot["snapshot_date"]}

PRIMARY OBJECTIVE

Answer all {len(questions)} questions as usefully and analytically as possible.

The answers will be generated repeatedly on different dates. Therefore, emphasize:

* new information;
* changes since the previous snapshot;
* current trends;
* recent company news;
* current market sentiment;
* identifiable risks and catalysts;
* financial implications;
* possible relevance to future stock performance.

EVIDENCE HIERARCHY

Use evidence in the following order:

1. Current Finnhub snapshot.
2. Comparison with the previous stored Finnhub snapshot.
3. Recent Finnhub company news.
4. Historical Finnhub financial metrics and earnings data.
5. Facts and figures explicitly stated in the question.
6. Stable general knowledge about the company and its business model.
7. Careful financial reasoning and inference.

IMPORTANT: Facts or figures explicitly included in a question may be treated as supplied contextual information. You may analyse their implications even when Finnhub does not independently repeat those figures.

ANSWERING RULES

You must provide a useful answer for every question.

Do not stop after saying that Finnhub lacks a specific metric.

When exact information is unavailable:

* State briefly which exact value is unavailable.
* Use the facts contained in the question as analytical inputs.
* Use related available Finnhub metrics, news or company information.
* Explain the most plausible financial implication.
* Clearly label the result as an inference, estimate, scenario analysis or directional assessment.
* Explain what additional evidence would be required for a definitive answer.

For example, instead of only writing:

“The supplied Finnhub data does not provide enough information.”

Write:

“Finnhub does not provide the segment-level figure needed to calculate the exact change. However, using the 36% prior growth rate stated in the question and Alphabet’s current profitability and recent cloud-related developments, continued growth near that rate would indicate strong scaling, while a material slowdown would weaken the investment case. This is a directional assessment rather than a verified current-year calculation.”

Do not invent exact financial values, dates, legal outcomes or company events.

Do not present an inference as a confirmed fact.

Do not assume that a missing metric equals zero.

Do not give personalised financial advice or an unsupported buy, sell or hold recommendation.

ANALYTICAL APPROACH

For each question, perform the following reasoning:

1. Identify the financial or strategic issue being tested.
2. Extract relevant facts from the question.
3. Find related evidence in the Finnhub snapshot and previous-snapshot comparison.
4. Use stable company knowledge where appropriate.
5. Determine whether the likely implication is positive, negative, mixed or neutral.
6. Explain the possible short-, medium- or long-term stock relevance.
7. State the confidence level of the assessment.

Where a numerical calculation is possible from information contained in the question or Finnhub data, calculate it explicitly.

Where the question asks for a projection, provide a scenario-based answer rather than refusing to answer.

Use scenarios such as:

* Positive scenario
* Base scenario
* Negative scenario

Only include all three scenarios when they materially improve the answer.

DAILY CHANGE ANALYSIS

For every question, check whether the following are relevant:

* current share-price change;
* current percentage price change;
* price movement since the previous stored snapshot;
* newly appearing company news;
* changed analyst recommendations;
* earnings surprises;
* profitability trends;
* valuation metrics;
* growth metrics;
* leverage and liquidity;
* regulatory or legal developments;
* acquisitions;
* capital expenditure;
* changes in market sentiment.

Do not force unrelated daily share-price data into every answer. Include it only when it helps answer the question.

OUTPUT FORMAT

Return exactly {len(questions)} numbered sections.

Reproduce every complete original question without shortening, combining or rewriting it.

Use the following format:

QUESTION 1 <complete original question>

ANSWER
<direct answer to the question in approximately 1–3 paragraphs>

AVAILABLE EVIDENCE

* <relevant Finnhub evidence>
* <relevant facts supplied in the question>
* <relevant previous-snapshot change>
* <relevant recent news>

ANALYSIS AND INFERENCE
<financial reasoning based on the evidence. Clearly distinguish verified facts from inference. When exact data is unavailable, provide a directional or scenario-based assessment rather than ending the answer.>

STOCK-PERFORMANCE RELEVANCE
<explain whether the evidence is likely positive, negative, mixed or neutral for future stock performance, and whether the effect is likely short-, medium- or long-term>

CONFIDENCE
<High, Medium or Low, followed by one sentence explaining why>

Repeat this structure for every question.

Do not add an introduction before QUESTION 1.

Do not add a conclusion after the final question.

QUESTIONS

{questions_text}

CURRENT FINNHUB SNAPSHOT

{serialise_json(current_snapshot)}

COMPARISON WITH THE PREVIOUS SNAPSHOT

{serialise_json(daily_comparison)}

""".strip()

In [12]:
def answer_questions_with_llm(
    prompt: str
) -> str:
    
    #Call the model with retry handling.
    
    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            result = llm_client.responses.create(
                model=LLM_MODEL,
                input=prompt
            )

            output_text = result.output_text

            if not output_text or not output_text.strip():
                raise RuntimeError(
                    "The LLM returned an empty response."
                )

            return output_text.strip()

        except Exception as error:
            if attempt == MAX_LLM_RETRIES:
                raise RuntimeError(
                    f"LLM request failed after {MAX_LLM_RETRIES} attempts: "
                    f"{error}"
                ) from error

            wait_time = min(
                60,
                (2 ** attempt) * 5 + random.uniform(0, 2)
            )

            print(
                f"    LLM request failed. "
                f"Retry {attempt}/{MAX_LLM_RETRIES} "
                f"after {wait_time:.1f} seconds."
            )

            time.sleep(wait_time)

    raise RuntimeError("The LLM request failed.")

In [13]:
def normalise_text_for_validation(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9 ]+", "", text)
    return text.strip()


def validate_llm_output(
    output_text: str,
    questions: list[str]
) -> list[int]:
    
    #Return question numbers whose complete text does not appear in the output.
   
    normalised_output = normalise_text_for_validation(output_text)
    missing_questions = []

    for index, question in enumerate(questions, start=1):
        normalised_question = normalise_text_for_validation(question)

        if normalised_question not in normalised_output:
            missing_questions.append(index)

    return missing_questions

In [14]:
def answer_file_path(
    company_name: str,
    answer_date: str
) -> Path:
    date_directory = ANSWERS_DIRECTORY / answer_date

    filename = (
        f"{safe_filename(company_name)}"
        f"_answer_{answer_date}.txt"
    )

    return date_directory / filename


def save_answer_file(
    company_name: str,
    ticker: str,
    category: str,
    answer_date: str,
    question_file: Path,
    answer_text: str,
    missing_question_numbers: list[int]
) -> Path:
    path = answer_file_path(
        company_name=company_name,
        answer_date=answer_date
    )

    path.parent.mkdir(parents=True, exist_ok=True)

    validation_status = (
        "PASS"
        if not missing_question_numbers
        else (
            "WARNING - complete text not detected for question numbers: "
            + ", ".join(map(str, missing_question_numbers))
        )
    )

    file_content = f"""COMPANY: {company_name}
TICKER: {ticker}
CATEGORY: {category}
ANALYSIS DATE: {answer_date}
QUESTION FILE: {question_file}
MODEL: {LLM_MODEL}
VALIDATION: {validation_status}

{"=" * 80}

{answer_text}
"""

    path.write_text(
        file_content,
        encoding="utf-8"
    )

    return path

In [15]:
def load_companies(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Company CSV was not found: {csv_path}"
        )

    companies = pd.read_csv(csv_path)

    # Support a few common alternative column names.
    column_aliases = {
        "name": "company_name",
        "company": "company_name",
        "security": "company_name",
        "symbol": "ticker",
        "sector": "category",
        "gics sector": "category",
        "gics_sector": "category"
    }

    rename_mapping = {}

    for column in companies.columns:
        normalised_column = str(column).strip().lower()

        if normalised_column in column_aliases:
            rename_mapping[column] = column_aliases[normalised_column]
        else:
            rename_mapping[column] = normalised_column

    companies = companies.rename(columns=rename_mapping)

    required_columns = {
        "company_name",
        "ticker",
        "category"
    }

    missing_columns = required_columns - set(companies.columns)

    if missing_columns:
        raise ValueError(
            "The company CSV is missing these columns: "
            + ", ".join(sorted(missing_columns))
        )

    companies = companies[
        ["company_name", "ticker", "category"]
    ].copy()

    companies = companies.dropna(
        subset=["company_name", "ticker"]
    )

    companies["company_name"] = (
        companies["company_name"].astype(str).str.strip()
    )

    companies["ticker"] = (
        companies["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    companies["category"] = (
        companies["category"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    companies = companies.drop_duplicates(
        subset=["ticker"],
        keep="first"
    )

    return companies.reset_index(drop=True)

In [16]:
def process_all_companies() -> pd.DataFrame:
    run_date = datetime.now().date().isoformat()

    print(f"Starting daily processing for {run_date}")
    print(f"LLM model: {LLM_MODEL}")
    print(f"Company CSV: {COMPANIES_CSV}")
    print(f"Questions directory: {QUESTIONS_DIRECTORY}")
    print()

    companies = load_companies(COMPANIES_CSV)

    if COMPANY_LIMIT is not None:
        companies = companies.head(COMPANY_LIMIT)

    question_file_index = build_question_file_index(
        QUESTIONS_DIRECTORY
    )

    results = []

    total_companies = len(companies)

    for row_index, row in companies.iterrows():
        company_number = row_index + 1

        company_name = row["company_name"]
        ticker = row["ticker"]
        category = row["category"]

        print("=" * 80)
        print(
            f"[{company_number}/{total_companies}] "
            f"{company_name} ({ticker})"
        )

        output_path = answer_file_path(
            company_name=company_name,
            answer_date=run_date
        )

        if output_path.exists() and not OVERWRITE_EXISTING_ANSWERS:
            print(f"Skipping: today's answer already exists at {output_path}")

            results.append({
                "company_name": company_name,
                "ticker": ticker,
                "status": "skipped_existing",
                "question_count": None,
                "answer_file": str(output_path),
                "snapshot_file": None,
                "error": None
            })

            continue

        try:
            # ------------------------------------------------
            # 1. Find and load questions
            # ------------------------------------------------

            question_file = find_question_file(
                company_name=company_name,
                ticker=ticker,
                question_file_index=question_file_index
            )

            if question_file is None:
                raise FileNotFoundError(
                    f"No question file was found for {company_name} "
                    f"({ticker})."
                )

            questions = load_questions(question_file)

            print(
                f"    Loaded {len(questions)} questions from "
                f"{question_file.name}"
            )

            if len(questions) != 30:
                print(
                    f"    Warning: expected 30 questions but found "
                    f"{len(questions)}."
                )

            # ------------------------------------------------
            # 2. Find previous snapshot before overwriting
            # ------------------------------------------------

            previous_snapshot = find_previous_snapshot(
                company_name=company_name,
                ticker=ticker,
                current_date=run_date
            )

            if previous_snapshot:
                print(
                    "    Previous snapshot found: "
                    f"{previous_snapshot.get('snapshot_date')}"
                )
            else:
                print("    No previous snapshot found.")

            # ------------------------------------------------
            # 3. Fetch and save today's Finnhub snapshot
            # ------------------------------------------------

            current_snapshot = fetch_finnhub_snapshot(
                company_name=company_name,
                ticker=ticker,
                category=category,
                snapshot_date=run_date
            )

            saved_snapshot_path = save_snapshot(
                current_snapshot
            )

            print(
                f"    Snapshot saved to {saved_snapshot_path}"
            )

            # ------------------------------------------------
            # 4. Calculate daily changes
            # ------------------------------------------------

            daily_comparison = compare_snapshots(
                current_snapshot=current_snapshot,
                previous_snapshot=previous_snapshot
            )

            # ------------------------------------------------
            # 5. Build prompt and call LLM
            # ------------------------------------------------

            prompt = build_company_prompt(
                company_name=company_name,
                ticker=ticker,
                category=category,
                questions=questions,
                current_snapshot=current_snapshot,
                daily_comparison=daily_comparison
            )

            print(
                f"    Prompt length: {len(prompt):,} characters"
            )
            print("    Sending questions the LLM...")

            answer_text = answer_questions_with_llm(prompt)

            # ------------------------------------------------
            # 6. Validate and save output
            # ------------------------------------------------

            missing_question_numbers = validate_llm_output(
                output_text=answer_text,
                questions=questions
            )

            if missing_question_numbers:
                print(
                    "    Warning: complete question text was not detected "
                    "for question numbers: "
                    + ", ".join(map(str, missing_question_numbers))
                )
            else:
                print(
                    "    Validation passed: all complete questions "
                    "were detected."
                )

            saved_answer_path = save_answer_file(
                company_name=company_name,
                ticker=ticker,
                category=category,
                answer_date=run_date,
                question_file=question_file,
                answer_text=answer_text,
                missing_question_numbers=missing_question_numbers
            )

            print(
                f"    Answer saved to {saved_answer_path}"
            )

            results.append({
                "company_name": company_name,
                "ticker": ticker,
                "status": "completed",
                "question_count": len(questions),
                "answer_file": str(saved_answer_path),
                "snapshot_file": str(saved_snapshot_path),
                "error": None
            })

        except Exception as error:
            print(f"    ERROR: {error}")

            results.append({
                "company_name": company_name,
                "ticker": ticker,
                "status": "failed",
                "question_count": None,
                "answer_file": None,
                "snapshot_file": None,
                "error": str(error)
            })

        print()

    results_dataframe = pd.DataFrame(results)

    log_directory = ANSWERS_DIRECTORY / run_date
    log_directory.mkdir(parents=True, exist_ok=True)

    log_path = log_directory / f"processing_log_{run_date}.csv"

    results_dataframe.to_csv(
        log_path,
        index=False
    )

    print("=" * 80)
    print("Daily processing finished.")
    print(f"Processing log: {log_path}")

    if not results_dataframe.empty:
        print()
        print(results_dataframe["status"].value_counts())

    return results_dataframe

In [17]:
results_df = process_all_companies()
results_df

Starting daily processing for 2026-07-15
LLM model: gpt-oss-120b
Company CSV: /mnt/primary/sp500.csv
Questions directory: /mnt/primary/questions

[1/100] Alphabet Inc. (Class A) (GOOGL)
    Loaded 30 questions from Alphabet Inc. (Class A).txt
    No previous snapshot found.
    Fetching quote...
    Fetching company_profile...
    Fetching basic_financials...
    Fetching company_news...
    Fetching recommendation_trends...
    Fetching earnings_surprises...
    Snapshot saved to /mnt/primary/finnhub_snapshots/2026-07-15/Alphabet Inc. (Class A)_GOOGL_snapshot.json
    Prompt length: 91,718 characters
    Sending questions the LLM...
    Answer saved to /mnt/primary/answers/2026-07-15/Alphabet Inc. (Class A)_answer_2026-07-15.txt

[2/100] AT&T (T)
    Loaded 30 questions from AT&T.txt
    No previous snapshot found.
    Fetching quote...
    Fetching company_profile...
    Fetching basic_financials...
    Fetching company_news...
    Fetching recommendation_trends...
    Fetching earni

,company_name,ticker,status,question_count,answer_file,snapshot_file,error
0,Alphabet Inc. (Class A),GOOGL,completed,30,/mnt/primary/answers/2026-07-15/Alphabet Inc. ...,/mnt/primary/finnhub_snapshots/2026-07-15/Alph...,None
1,AT&T,T,completed,30,/mnt/primary/answers/2026-07-15/AT&T_answer_20...,/mnt/primary/finnhub_snapshots/2026-07-15/AT&T...,None
2,Electronic Arts,EA,completed,30,/mnt/primary/answers/2026-07-15/Electronic Art...,/mnt/primary/finnhub_snapshots/2026-07-15/Elec...,None
3,Meta Platforms,META,completed,30,/mnt/primary/answers/2026-07-15/Meta Platforms...,/mnt/primary/finnhub_snapshots/2026-07-15/Meta...,None
4,Netflix,NFLX,completed,30,/mnt/primary/answers/2026-07-15/Netflix_answer...,/mnt/primary/finnhub_snapshots/2026-07-15/Netf...,None
...,...,...,...,...,...,...,...
95,Constellation Energy,CEG,completed,30,/mnt/primary/answers/2026-07-15/Constellation ...,/mnt/primary/finnhub_snapshots/2026-07-15/Cons...,None
96,Dominion Energy,D,completed,30,/mnt/primary/answers/2026-07-15/Dominion Energ...,/mnt/primary/finnhub_snapshots/2026-07-15/Domi...,None
97,Vistra Corp.,VST,completed,30,/mnt/primary/answers/2026-07-15/Vistra Corp_an...,/mnt/primary/finnhub_snapshots/2026-07-15/Vist...,None
98,WEC Energy Group,WEC,completed,30,/mnt/primary/answers/2026-07-15/WEC Energy Gro...,/mnt/primary/finnhub_snapshots/2026-07-15/WEC ...,None
